# CodeLab Notebook: Self-Consistency Prompting với LM Studio

Notebook này minh hoạ một pipeline hoàn chỉnh để:
- gọi một local LLM qua giao thức OpenAI-compatible của LM Studio,
- lấy nhiều lời giải độc lập cho cùng một bài toán,
- dùng embedding để gom các câu trả lời gần nghĩa,
- chọn ra đáp án nhất quán nhất làm kết quả cuối cùng.

Bài toán mẫu là một tình huống suy luận theo chính sách hoàn tiền, nơi có **quy tắc chung** và **ngoại lệ đặc biệt**. Đây là dạng bài phù hợp để quan sát giá trị của self-consistency vì model có thể đi theo nhiều hướng suy luận khác nhau trước khi hội tụ.

## Self-Consistency Prompting là gì?

Self-consistency là một kỹ thuật prompt engineering dành cho các bài toán cần suy luận nhiều bước. Thay vì tin vào một chain-of-thought duy nhất, ta:
- lấy nhiều mẫu trả lời từ cùng một prompt bằng cách tăng `temperature`,
- coi mỗi mẫu là một "đường suy luận" độc lập,
- tổng hợp các kết quả để chọn ra đáp án xuất hiện nhất quán nhất.

Về mặt kỹ thuật, self-consistency thường hiệu quả khi:
- bài toán có nhiều bước suy luận và dễ lệch ở giữa chừng,
- model có thể diễn đạt cùng một kết luận theo nhiều cách khác nhau,
- ta muốn giảm rủi ro do một mẫu ngẫu nhiên bị suy luận sai.

Trong notebook này, bước tổng hợp không dùng so khớp chuỗi thô. Thay vào đó, ta dùng **embedding + cosine similarity** để nhóm các câu trả lời gần nghĩa, kể cả khi chúng dùng từ khác nhau.

## Sơ đồ trực quan

<div align="center">
  <img src="assets/self_consistency/01-branching-paths.png" width="92%" alt="Một prompt tạo ra nhiều mẫu suy luận" />
  <p><em>1. Một prompt được mở rộng thành nhiều mẫu suy luận độc lập.</em></p>
  <img src="assets/self_consistency/02-embedding-similarity.png" width="92%" alt="Embedding và cosine similarity" />
  <p><em>2. Mỗi response được embed, so sánh bằng cosine similarity, rồi gom theo ngữ nghĩa.</em></p>
  <img src="assets/self_consistency/03-majority-vote.png" width="92%" alt="Majority vote và final answer" />
  <p><em>3. Chọn nhóm đồng thuận mạnh nhất để lấy đáp án cuối cùng.</em></p>
</div>

## Chuẩn bị môi trường

Điều kiện để notebook chạy được:
- LM Studio đang chạy local server tại `http://localhost:1234`
- model chat đang được load trong LM Studio
- model embedding cũng đang được load trong LM Studio

Nếu môi trường Python của bạn chưa có thư viện, chạy cell cài đặt bên dưới một lần.

In [8]:
# Bỏ comment nếu môi trường của bạn chưa có thư viện cần thiết
# %pip install -q openai numpy scikit-learn


In [14]:
from openai import OpenAI
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
API_KEY = "lm-studio"  # LM Studio không cần API key thật
CHAT_MODEL = "gemma-4-e4b-it"
EMBEDDING_MODEL = "text-embedding-nomic-embed-text-v1.5"

TEMPERATURE = 0.7
NUM_SAMPLES = 5
MAX_TOKENS = 512
SIMILARITY_THRESHOLD = 0.85

def build_client(base_url=LM_STUDIO_BASE_URL):
    return OpenAI(base_url=base_url, api_key=API_KEY)

client = build_client()
print(f"Đã kết nối LM Studio qua {LM_STUDIO_BASE_URL}")


Đã kết nối LM Studio qua http://localhost:1234/v1


## Thiết kế prompt thử nghiệm

Prompt bên dưới cố tình chứa:
- quy tắc chuẩn: quá 30 ngày thì không hoàn tiền,
- ngoại lệ mạnh: nếu có `Critical Bug` gây mất dữ liệu thì vẫn được hoàn 100%.

Đây là tình huống tốt để test self-consistency vì model phải nhận ra rằng **ngoại lệ đặc biệt ghi đè quy tắc chung**. Nếu suy luận không chắc chắn, một số mẫu có thể bám vào mốc 30 ngày và trả lời sai.

In [15]:
SYSTEM_PROMPT = (
    "You are a helpful assistant. Keep your explanations clear, structured, "
    "and concise. End with one short final conclusion sentence."
)

CASE_PROMPT = """Bạn là chuyên viên xử lý khiếu nại khách hàng. Hãy đọc kỹ chính sách sau:

CHÍNH SÁCH HOÀN TIỀN:
1. Hoàn tiền 100% nếu yêu cầu trong vòng 30 ngày và thời gian sử dụng phần mềm dưới 10 giờ.
2. Hoàn 50% bằng Credit nếu quá 10 giờ sử dụng nhưng vẫn trong vòng 30 ngày.
3. KHÔNG hoàn tiền nếu đã mua quá 30 ngày.
4. NGOẠI LỆ ĐẶC BIỆT: Bất kể thời gian mua hay số giờ sử dụng, nếu phần mềm gặp lỗi hệ thống nghiêm trọng (Critical Bug) từ phía chúng tôi gây mất dữ liệu của khách, khách hàng được quyền yêu cầu hoàn tiền 100%.

EMAIL KHÁCH HÀNG:
"Chào các bạn, tôi mua phần mềm này 45 ngày trước. Tôi mới dùng được khoảng 12 tiếng thôi. Hôm qua sau khi tải bản cập nhật mới nhất của các bạn, toàn bộ file thiết kế của tôi bị xóa sạch, phần mềm crash liên tục và giờ không thể mở lên được nữa. Tôi rất tức giận và muốn lấy lại tiền ngay lập tức!"

Dựa trên chính sách và email, khách hàng này sẽ được giải quyết như thế nào? Hãy suy luận và đưa ra kết luận cuối cùng trong 1 câu ngắn gọn."""

def generate_completion(prompt, temperature=TEMPERATURE):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=MAX_TOKENS,
    )
    return response.choices[0].message.content.strip()

def collect_reasoning_paths(prompt, num_samples=NUM_SAMPLES, temperature=TEMPERATURE):
    samples = []
    for sample_id in range(1, num_samples + 1):
        answer = generate_completion(prompt, temperature=temperature)
        samples.append(answer)

        print(f"--- Mẫu suy luận {sample_id} ---")
        print(answer)
        print("-" * 80)

    return samples


## 1. Một lần gọi model làm baseline

Trước tiên, ta lấy một câu trả lời duy nhất để làm mốc so sánh. Đây là cách prompt thông thường hoạt động: model suy luận một lần và ta chấp nhận luôn kết quả đó.

In [16]:
baseline_answer = generate_completion(CASE_PROMPT, temperature=TEMPERATURE)

print("=== Baseline answer ===")
print(baseline_answer)


=== Baseline answer ===
Chào bạn, với vai trò chuyên viên xử lý khiếu nại, tôi xin phân tích tình huống dựa trên Chính sách Hoàn tiền và nội dung email của khách hàng như sau:

**Phân tích Tình huống:**

1. **Thời gian mua:** Khách hàng mua phần mềm cách đây 45 ngày.
2. **Thời gian sử dụng:** Khoảng 12 giờ.
3. **Vấn đề kỹ thuật (Điểm mấu chốt):** Toàn bộ file thiết kế bị xóa sạch sau khi cập nhật, phần mềm crash liên tục và không thể mở được.

**Đối chiếu với Chính sách Hoàn tiền:**

*   **Điều khoản 1 & 2 (Giới hạn thời gian/giờ sử dụng):** Khách hàng đã mua quá 30 ngày ($\rightarrow$ Không đủ điều kiện theo Mục 3).
*   **Điều khoản 3 (Hết thời hạn):** Việc mua cách đây 45 ngày vi phạm quy định chung.
*   **Điều khoản 4 (Ngoại lệ Đặc biệt):** Chính sách nêu rõ: "Bất kể thời gian mua hay số giờ sử dụng, nếu phần mềm gặp lỗi hệ thống nghiêm trọng (Critical Bug) từ phía chúng tôi gây mất dữ liệu của khách, khách hàng được quyền yêu cầu hoàn tiền 100%."

**Kết luận xử lý:**

Mặc dù khách 

## 2. Lấy nhiều đường suy luận độc lập

Ở bước này, ta gọi model nhiều lần với cùng prompt.
- `temperature > 0` giúp model tạo ra các hướng diễn giải hơi khác nhau.
- nếu đa số các mẫu hội tụ về cùng một kết luận, ta có tín hiệu rằng đáp án đó ổn định hơn.
- nếu các mẫu phân tán mạnh, đó là dấu hiệu bài toán còn mơ hồ hoặc prompt chưa đủ rõ.

Số mẫu `NUM_SAMPLES = 5` là mức nhỏ, đủ để demo. Với tác vụ khó hơn, có thể tăng lên `7`, `10` hoặc `20` tuỳ chi phí và độ trễ chấp nhận được.

In [12]:
responses = collect_reasoning_paths(CASE_PROMPT)


--- Mẫu suy luận 1 ---
Chào bạn, với vai trò là chuyên viên xử lý khiếu nại, tôi xin phân tích tình huống của khách hàng dựa trên Chính sách hoàn tiền đã cung cấp:

**Phân tích theo các điều khoản:**

1. **Điều kiện thời gian mua (30 ngày):** Khách hàng mua phần mềm cách đây 45 ngày. Theo Điều khoản 3 ("KHÔNG hoàn tiền nếu đã mua quá 30 ngày"), về mặt nguyên tắc chung, yêu cầu hoàn tiền của khách hàng sẽ bị từ chối.
2. **Điều kiện sử dụng:** Thời gian sử dụng là 12 giờ (vượt qua mốc 10 giờ). Theo Điều khoản 2, nếu nằm trong khung thời gian cho phép, họ sẽ được hoàn 50% bằng Credit. Tuy nhiên, điều này không áp dụng do vi phạm điều kiện thời gian mua.
3. **Điều khoản Ngoại lệ Đặc biệt (Critical Bug):** Khách hàng nêu rõ vấn đề kỹ thuật nghiêm trọng: "Hôm qua sau khi tải bản cập nhật mới nhất của các bạn, toàn bộ file thiết kế của tôi bị xóa sạch, phần mềm crash liên tục và giờ không thể mở lên được nữa." Tình huống này mô tả chính xác một lỗi hệ thống nghiêm trọng (Critical Bug) từ phía

## 3. Semantic voting bằng embedding

Nếu chỉ dùng so khớp chuỗi chính xác, các câu trả lời kiểu:
- "Khách hàng được hoàn tiền 100% do critical bug"
- "Nên refund toàn bộ vì có lỗi nghiêm trọng gây mất dữ liệu"

sẽ bị xem là khác nhau dù ý nghĩa gần như trùng nhau.

Vì vậy ta:
1. chuyển từng câu trả lời thành embedding,
2. tính cosine similarity giữa mọi cặp câu trả lời,
3. gom các câu đủ giống nhau vào cùng một nhóm,
4. chọn nhóm lớn nhất làm "phe đa số".

Để notebook dễ đọc, phần code bên dưới dùng một cách nhóm rất đơn giản: lấy từng câu trả lời chưa được xếp nhóm làm mốc, rồi kéo các câu đủ giống với nó vào cùng nhóm.

Trong nhóm thắng, notebook chọn câu có độ tương đồng trung bình cao nhất với các câu còn lại làm câu đại diện.

In [13]:
def embed_texts(texts):
    vectors = []
    for text in texts:
        response = client.embeddings.create(
            input=text,
            model=EMBEDDING_MODEL,
        )
        vectors.append(response.data[0].embedding)
    return np.array(vectors)

def build_similarity_matrix(embedding_matrix):
    return cosine_similarity(embedding_matrix)

def group_similar_responses(similarity_matrix, threshold=SIMILARITY_THRESHOLD):
    groups = []
    used_indices = set()

    for anchor_idx in range(len(similarity_matrix)):
        if anchor_idx in used_indices:
            continue

        current_group = [anchor_idx]
        used_indices.add(anchor_idx)

        for other_idx in range(anchor_idx + 1, len(similarity_matrix)):
            is_similar = similarity_matrix[anchor_idx, other_idx] >= threshold
            if other_idx not in used_indices and is_similar:
                current_group.append(other_idx)
                used_indices.add(other_idx)

        groups.append(current_group)

    return groups

def pick_representative_response(group, similarity_matrix, texts):
    best_idx = max(
        group,
        key=lambda idx: similarity_matrix[idx, group].mean(),
    )
    return best_idx, texts[best_idx]

response_vectors = embed_texts(responses)
similarity_matrix = build_similarity_matrix(response_vectors)
response_groups = group_similar_responses(
    similarity_matrix,
    threshold=SIMILARITY_THRESHOLD,
)
best_group = max(response_groups, key=len)
best_response_idx, final_answer = pick_representative_response(
    best_group,
    similarity_matrix,
    responses,
)
confidence = len(best_group) / len(responses)

print("=== Cosine similarity matrix ===")
print(np.round(similarity_matrix, 2))
print()

print("=== Các nhóm câu trả lời ===")
for group_id, group in enumerate(response_groups, start=1):
    print(f"Nhóm {group_id}:")
    for response_idx in group:
        print(f"- Mẫu {response_idx + 1}: {responses[response_idx]}")
    print()

print("=== Kết quả self-consistency ===")
print(f"Số phiếu đồng thuận: {len(best_group)}/{len(responses)}")
print(f"Độ tự tin ước lượng: {confidence:.0%}")
print(f"Câu đại diện được chọn: mẫu {best_response_idx + 1}")
print(final_answer)


=== Cosine similarity matrix ===
[[1.   0.98 0.97 0.97 0.98]
 [0.98 1.   0.99 0.98 0.99]
 [0.97 0.99 1.   0.99 0.98]
 [0.97 0.98 0.99 1.   0.99]
 [0.98 0.99 0.98 0.99 1.  ]]

=== Các nhóm câu trả lời ===
Nhóm 1:
- Mẫu 1: Chào bạn, với vai trò là chuyên viên xử lý khiếu nại, tôi xin phân tích tình huống của khách hàng dựa trên Chính sách hoàn tiền đã cung cấp:

**Phân tích theo các điều khoản:**

1. **Điều kiện thời gian mua (30 ngày):** Khách hàng mua phần mềm cách đây 45 ngày. Theo Điều khoản 3 ("KHÔNG hoàn tiền nếu đã mua quá 30 ngày"), về mặt nguyên tắc chung, yêu cầu hoàn tiền của khách hàng sẽ bị từ chối.
2. **Điều kiện sử dụng:** Thời gian sử dụng là 12 giờ (vượt qua mốc 10 giờ). Theo Điều khoản 2, nếu nằm trong khung thời gian cho phép, họ sẽ được hoàn 50% bằng Credit. Tuy nhiên, điều này không áp dụng do vi phạm điều kiện thời gian mua.
3. **Điều khoản Ngoại lệ Đặc biệt (Critical Bug):** Khách hàng nêu rõ vấn đề kỹ thuật nghiêm trọng: "Hôm qua sau khi tải bản cập nhật mới nhất 

## 4. Ghi chú kỹ thuật và best practices

Một vài điểm quan trọng khi áp dụng self-consistency trong thực tế:
- Chỉ hiệu quả khi bài toán thực sự cần suy luận. Với tác vụ truy xuất fact đơn giản, lấy nhiều mẫu thường chỉ làm tăng chi phí.
- Nên ép model trả lời theo format ổn định. Ví dụ: "Phân tích ngắn", rồi "Kết luận cuối: ...". Format ổn định giúp bước gom cụm và hậu xử lý dễ hơn.
- `temperature` phải đủ để tạo đa dạng, nhưng không quá cao đến mức câu trả lời mất chất lượng. Thực tế thường thử trong khoảng `0.4 -> 0.8`.
- Có thể dùng nhiều chiến lược tổng hợp: exact match, embedding clustering, một LLM khác đóng vai trò judge/ranker, hoặc rule-based parser nếu đầu ra có cấu trúc.
- Ngưỡng `SIMILARITY_THRESHOLD` cần được hiệu chỉnh theo bài toán. Ngưỡng cao quá sẽ tách các câu paraphrase thành nhiều cụm; thấp quá sẽ gom cả các câu khác nghĩa.
- Khi cụm thắng quá nhỏ hoặc phân tán mạnh, đó là tín hiệu nên xem lại prompt, thêm context, hoặc fallback sang human review.

Trong case refund này, nếu pipeline hoạt động đúng, cụm thắng nên kết luận rằng khách hàng **được hoàn tiền 100%** vì lỗi nghiêm trọng gây mất dữ liệu là ngoại lệ có ưu tiên cao hơn quy tắc "quá 30 ngày thì không hoàn".